<a href="https://colab.research.google.com/github/wa255007/helloworld/blob/master/run_all_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#!/usr/bin/env python3
"""
BLEVE Blast Loading Prediction - Complete Pipeline
Optimized for Mac GPU (MPS) with all 4 models
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
import os
import time
import gc
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, QuantileTransformer, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# LightGBM
import lightgbm as lgb

print("="*70)
print("BLEVE BLAST LOADING PREDICTION - COMPLETE PIPELINE")
print("="*70)

# Check device - use CPU for stability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("✓ Using CPU (stable mode)")

# Disable MPS due to stability issues on some Macs
if torch.backends.mps.is_available():
    print("  Note: MPS available but disabled for stability")

print(f"✓ PyTorch version: {torch.__version__}")
print("="*70)

def calculate_metrics(y_true, y_pred):
    """Calculate all metrics including MAPE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAPE': mape, 'R2': r2}

def save_submission(test_preds, model_name, scaler_y):
    """Save predictions in Kaggle submission format."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_{model_name}_{timestamp}.csv"

    # Load test IDs
    test_ids = np.arange(len(test_preds))

    # Inverse transform
    if scaler_y is not None:
        test_preds_orig = scaler_y.inverse_transform(test_preds.reshape(-1, 1)).flatten()
    else:
        test_preds_orig = test_preds

    # Ensure no negative values
    test_preds_orig = np.maximum(test_preds_orig, 0.001)

    # Create submission
    submission = pd.DataFrame({
        'id': test_ids,
        'Target Pressure (bar)': test_preds_orig
    })
    submission.to_csv(filename, index=False)
    print(f"  ✓ Saved: {filename}")
    return filename

# ============================================================================
# STEP 1: LOAD AND PREPROCESS DATA
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOADING AND PREPROCESSING DATA")
print("="*70)

print("\nLoading train(2).csv...")
train_df = pd.read_csv("train(2).csv", index_col=0)
print(f"  ✓ Training data: {train_df.shape}")

print("\nLoading test(2).csv...")
test_df = pd.read_csv("test(2).csv", index_col=0)
print(f"  ✓ Test data: {test_df.shape}")

# Display target info
print(f"\n  Target variable statistics:")
print(f"    Mean: {train_df['Target Pressure (bar)'].mean():.4f}")
print(f"    Std:  {train_df['Target Pressure (bar)'].std():.4f}")
print(f"    Min:  {train_df['Target Pressure (bar)'].min():.4f}")
print(f"    Max:  {train_df['Target Pressure (bar)'].max():.4f}")

from sklearn.impute import SimpleImputer

numeric_cols = train_df.select_dtypes(include=[np.number]).columns
num_imputer = SimpleImputer(strategy="median")
train_df[numeric_cols] = num_imputer.fit_transform(train_df[numeric_cols])

def create_engineered_features(df):
    """
    Create new features based on domain knowledge and physical relationships.
    """
    df = df.copy()
    original_cols = set(df.columns)

    # ==================== TANK FEATURES ====================
    # Tank volume (cubic meters)
    df['Tank_Volume'] = (df['Tank Width (m)'] *
                         df['Tank Length (m)'] *
                         df['Tank Height (m)'])

    # Tank surface area (for heat transfer calculations)
    df['Tank_Surface_Area'] = 2 * (
        df['Tank Width (m)'] * df['Tank Length (m)'] +
        df['Tank Width (m)'] * df['Tank Height (m)'] +
        df['Tank Length (m)'] * df['Tank Height (m)']
    )

    # Tank aspect ratios (shape affects explosion dynamics)
    df['Tank_Width_to_Length_Ratio'] = df['Tank Width (m)'] / df['Tank Length (m)']
    df['Tank_Height_to_Width_Ratio'] = df['Tank Height (m)'] / df['Tank Width (m)']

    # Tank cross-sectional area
    df['Tank_Cross_Section_Area'] = df['Tank Width (m)'] * df['Tank Height (m)']

    # ==================== VAPOUR/LIQUID FEATURES ====================
    # Vapour fraction
    df['Vapour_Fraction'] = df['Vapour Height (m)'] / df['Tank Height (m)']

    # Liquid fraction (complement of vapour fraction)
    df['Liquid_Fraction'] = 1 - df['Vapour_Fraction']

    # Vapour volume
    df['Vapour_Volume'] = (df['Tank Width (m)'] *
                           df['Tank Length (m)'] *
                           df['Vapour Height (m)'])

    # # Liquid volume
    df['Liquid_Volume'] = df['Tank_Volume'] - df['Vapour_Volume']

    # Energy potential indicators
    df['Vapour_to_Liquid_Ratio'] = df['Vapour_Volume'] / (df['Liquid_Volume'] + 1e-6)

    # BLEVE height ratio
    df['BLEVE_Height_Ratio'] = df['BLEVE Height (m)'] / df['Tank Height (m)']

    # ==================== TEMPERATURE FEATURES ====================
    # Temperature differences (indicate thermal gradients)
    df['Temperature_Diff'] = df['Vapour Temperature (K)'] - df['Liquid Temperature (K)']

    # Distance from critical temperature
    df['Temp_Diff_to_Critical'] = (df['Liquid Critical Temperature (K)'] -
                                   df['Liquid Temperature (K)'])

    # Temperature ratio
    df['Temp_Ratio_Vapour_to_Liquid'] = (df['Vapour Temperature (K)'] /
                                          df['Liquid Temperature (K)'])

    # Boiling temperature difference
    df['Temp_Above_Boiling'] = (df['Liquid Temperature (K)'] -
                                df['Liquid Boiling Temperature (K)'])

    # ==================== PRESSURE FEATURES ====================
    # Pressure ratios (dimensionless indicators)
    df['Pressure_to_Critical_Ratio'] = (df['Tank Failure Pressure (bar)'] /
                                        df['Liquid Critical Pressure (bar)'])

    df['Pressure_Temp_Ratio'] = (df['Tank Failure Pressure (bar)'] /
                                 df['Liquid Temperature (K)'])

    # ==================== OBSTACLE FEATURES ====================
    # Obstacle volume
    df['Obstacle_Volume'] = (df['Obstacle Width (m)'] *
                             df['Obstacle Height (m)'] *
                             df['Obstacle Thickness (m)'])

    # Obstacle surface area facing blast
    df['Obstacle_Frontal_Area'] = df['Obstacle Width (m)'] * df['Obstacle Height (m)']

    # Obstacle aspect ratios
    df['Obstacle_Height_to_Width_Ratio'] = (df['Obstacle Height (m)'] /
                                            df['Obstacle Width (m)'])
    df['Obstacle_Thickness_to_Width_Ratio'] = (df['Obstacle Thickness (m)'] /
                                               df['Obstacle Width (m)'])

    # Distance features (blast wave attenuation)
    df['Inverse_Distance'] = 1 / (df['Obstacle Distance to BLEVE (m)'] + 1e-6)
    df['Distance_Squared'] = df['Obstacle Distance to BLEVE (m)'] ** 2
    df['Distance_Cubed'] = df['Obstacle Distance to BLEVE (m)'] ** 3

    # Angle in radians (for trigonometric calculations)
    df['Obstacle_Angle_Rad'] = np.deg2rad(df['Obstacle Angle'])
    df['Obstacle_Angle_Sin'] = np.sin(df['Obstacle_Angle_Rad'])
    df['Obstacle_Angle_Cos'] = np.cos(df['Obstacle_Angle_Rad'])

    # ==================== SENSOR FEATURES ====================
    # Sensor distance from origin (magnitude of position vector)
    df['Sensor_Distance_from_Origin'] = np.sqrt(
        df['Sensor Position x']**2 +
        df['Sensor Position y']**2 +
        df['Sensor Position z']**2
    )

    # Distance from BLEVE to sensor (approximate)
    df['Sensor_Distance_from_BLEVE'] = np.sqrt(
        (df['Sensor Position x'] - df['Obstacle Distance to BLEVE (m)'])**2 +
        df['Sensor Position y']**2 +
        df['Sensor Position z']**2
    )

    # Sensor position squared terms (for non-linear relationships)
    df['Sensor_Position_x_Squared'] = df['Sensor Position x'] ** 2
    df['Sensor_Position_y_Squared'] = df['Sensor Position y'] ** 2
    df['Sensor_Position_z_Squared'] = df['Sensor Position z'] ** 2

    # ==================== COMBINED FEATURES ====================
    # Substance temperature range
    df['Substance_Temp_Range'] = (df['Liquid Critical Temperature (K)'] -
                                  df['Liquid Boiling Temperature (K)'])

    # Energy indicator (pressure * volume)
    df['Energy_Indicator'] = df['Tank Failure Pressure (bar)'] * df['Tank_Volume']

    # Blast loading indicator
    df['Blast_Loading_Indicator'] = (df['Tank Failure Pressure (bar)'] *
                                     df['Vapour_Fraction'] /
                                     (df['Obstacle Distance to BLEVE (m)'] + 1))

    # Identify new features
    new_features = set(df.columns) - original_cols

    return df, list(new_features)

# Apply feature engineering
print("\n" + "=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

train_engineered, _ = create_engineered_features(train_df)
test_engineered, _ = create_engineered_features(test_df)

# Preprocess
def preprocess_data(df, is_train=True, scaler_X=None, scaler_y=None, le_status=None):
    df = df.copy()

    if is_train:
        y = df['Target Pressure (bar)'].values
        X = df.drop('Target Pressure (bar)', axis=1)
    else:
        X = df
        y = None

    # Encode categorical: Status
    if le_status is None:
        le_status = LabelEncoder()
        X['Status'] = le_status.fit_transform(X['Status'])
    else:
        X['Status'] = le_status.transform(X['Status'])

    # Drop non-predictive columns
    cols_to_drop = ['Sensor Position Side', 'Sensor Position x', 'Sensor Position y', 'Sensor Position z', 'Sensor ID']
    cols_to_drop = [c for c in cols_to_drop if c in X.columns]
    X = X.drop(columns=cols_to_drop)

    feature_names = X.columns.tolist()
    X = X.values.astype(np.float32)

    # Scale features
    if scaler_X is None:
        scaler_X = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
    else:
        X_scaled = scaler_X.transform(X)

    # Scale target
    if is_train:
        if scaler_y is None:
            scaler_y = QuantileTransformer(output_distribution='normal', random_state=42)
            y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
        else:
            y_scaled = scaler_y.transform(y.reshape(-1, 1)).flatten()
    else:
        y_scaled = None

    return X_scaled, y_scaled, y, scaler_X, scaler_y, le_status, feature_names

# Process training data
print("\nPreprocessing training data...")
X_full, y_full_scaled, y_full_orig, scaler_X, scaler_y, le_status, feature_names = preprocess_data(
    train_engineered, is_train=True
)
print(f"  ✓ Features: {len(feature_names)}")
print(f"  ✓ Feature names: {feature_names}")

# Split: 80% train, 20% validation
X_train, X_val, y_train_scaled, y_val_scaled, y_train_orig, y_val_orig = train_test_split(
    X_full, y_full_scaled, y_full_orig, test_size=0.2, random_state=42
)

print(f"\n  Data split:")
print(f"    Training:   {X_train.shape[0]:,} samples")
print(f"    Validation: {X_val.shape[0]:,} samples")

# Process test data
print("\nPreprocessing test data...")
X_test, _, _, _, _, _, _ = preprocess_data(
    test_engineered, is_train=False, scaler_X=scaler_X, scaler_y=scaler_y, le_status=le_status
)
print(f"  ✓ Test: {X_test.shape[0]:,} samples")

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_train_tensor = torch.FloatTensor(y_train_scaled).to(device)
y_val_tensor = torch.FloatTensor(y_val_scaled).to(device)

# DataLoader
batch_size = 1024
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

# ============================================================================
# STEP 2: TRAIN ALL MODELS
# ============================================================================
print("\n" + "="*70)
print("STEP 2: TRAINING MODELS")
print("="*70)

results = []

# -----------------------------------------------------------------------------
# MODEL 1: LightGBM
# -----------------------------------------------------------------------------
print("\n" + "-"*70)
print("MODEL 1: LightGBM")
print("-"*70)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 100,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
}

train_data = lgb.Dataset(X_train, label=y_train_scaled)
val_data = lgb.Dataset(X_val, label=y_val_scaled, reference=train_data)

start_time = time.time()
lgb_model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=2000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
)
lgb_time = time.time() - start_time

# Predictions
y_pred_lgb_val_scaled = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
y_pred_lgb_test_scaled = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)

y_pred_lgb_val = scaler_y.inverse_transform(y_pred_lgb_val_scaled.reshape(-1, 1)).flatten()

lgb_metrics = calculate_metrics(y_val_orig, y_pred_lgb_val)

print(f"\n  ✓ Training time: {lgb_time:.1f}s")
print(f"    RMSE: {lgb_metrics['RMSE']:.4f}")
print(f"    MAPE: {lgb_metrics['MAPE']:.2f}%")
print(f"    R²:   {lgb_metrics['R2']:.4f}")

lgb_file = save_submission(y_pred_lgb_test_scaled, 'lightgbm', scaler_y)
results.append(['LightGBM', lgb_metrics['RMSE'], lgb_metrics['MAPE'], lgb_metrics['R2'], lgb_time])


BLEVE BLAST LOADING PREDICTION - COMPLETE PIPELINE
✓ Using CPU (stable mode)
✓ PyTorch version: 2.10.0+cpu

STEP 1: LOADING AND PREPROCESSING DATA

Loading train(2).csv...
  ✓ Training data: (10050, 24)

Loading test(2).csv...
  ✓ Test data: (3203, 23)

  Target variable statistics:
    Mean: 0.3618
    Std:  0.5055
    Min:  0.0161
    Max:  9.1705

FEATURE ENGINEERING

Preprocessing training data...
  ✓ Features: 53
  ✓ Feature names: ['Tank Failure Pressure (bar)', 'Liquid Ratio', 'Tank Width (m)', 'Tank Length (m)', 'Tank Height (m)', 'BLEVE Height (m)', 'Vapour Height (m)', 'Vapour Temperature (K)', 'Liquid Temperature (K)', 'Obstacle Distance to BLEVE (m)', 'Obstacle Width (m)', 'Obstacle Height (m)', 'Obstacle Thickness (m)', 'Obstacle Angle', 'Status', 'Liquid Critical Pressure (bar)', 'Liquid Boiling Temperature (K)', 'Liquid Critical Temperature (K)', 'Tank_Volume', 'Tank_Surface_Area', 'Tank_Width_to_Length_Ratio', 'Tank_Height_to_Width_Ratio', 'Tank_Cross_Section_Area', 'Va

In [12]:

# -----------------------------------------------------------------------------
# NEURAL NETWORK MODELS
# -----------------------------------------------------------------------------

def train_nn_model(model, train_loader, X_val, y_val, y_val_orig, scaler_y,
                   optimizer, criterion, n_epochs=500, patience=20):
    """Train neural network with early stopping."""
    model = model.to(device)
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    # Initialize GradScaler for mixed precision training if using CUDA
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    for epoch in range(n_epochs):
        # Training
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()

            if scaler: # Mixed precision training
                with torch.cuda.amp.autocast():
                    y_pred = model(X_batch)
                    loss = criterion(y_pred, y_batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else: # Standard training (CPU or if scaler not initialized)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                loss.backward()
                optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            if scaler: # Mixed precision validation
                with torch.cuda.amp.autocast():
                    y_val_pred_scaled = model(X_val).cpu().numpy()
            else:
                y_val_pred_scaled = model(X_val).cpu().numpy()

            y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()
            val_loss = mean_squared_error(y_val_orig, y_val_pred)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        if (epoch + 1) % 100 == 0:
            print(f"      Epoch {epoch+1}/{n_epochs}, Val RMSE: {np.sqrt(val_loss):.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

# -----------------------------------------------------------------------------
# MODEL 2: MLP
# -----------------------------------------------------------------------------
print("\n" + "-"*70)
print("MODEL 2: MLP (Multi-Layer Perceptron)")
print("-"*70)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

mlp_model = MLP(X_train.shape[1], hidden_dims=[512, 256, 128], dropout_rate=0.2).to(device)
optimizer = optim.AdamW(mlp_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

start_time = time.time()
mlp_model = train_nn_model(mlp_model, train_loader, X_val_tensor, y_val_tensor, y_val_orig,
                          scaler_y, optimizer, criterion, n_epochs=500, patience=20)
mlp_time = time.time() - start_time

mlp_model.eval()
with torch.no_grad():
    y_pred_mlp_val_scaled = mlp_model(X_val_tensor).cpu().numpy()
    y_pred_mlp_test_scaled = mlp_model(X_test_tensor).cpu().numpy()

y_pred_mlp_val = scaler_y.inverse_transform(y_pred_mlp_val_scaled.reshape(-1, 1)).flatten()

mlp_metrics = calculate_metrics(y_val_orig, y_pred_mlp_val)

print(f"\n  ✓ Training time: {mlp_time:.1f}s")
print(f"    RMSE: {mlp_metrics['RMSE']:.4f}")
print(f"    MAPE: {mlp_metrics['MAPE']:.2f}%")
print(f"    R²:   {mlp_metrics['R2']:.4f}")

mlp_file = save_submission(y_pred_mlp_test_scaled, 'mlp', scaler_y)
results.append(['MLP', mlp_metrics['RMSE'], mlp_metrics['MAPE'], mlp_metrics['R2'], mlp_time])

# Clear memory
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()



----------------------------------------------------------------------
MODEL 2: MLP (Multi-Layer Perceptron)
----------------------------------------------------------------------

  ✓ Training time: 44.2s
    RMSE: 0.2183
    MAPE: 21.75%
    R²:   0.8403
  ✓ Saved: submission_mlp_20260419_184136.csv


In [13]:

# -----------------------------------------------------------------------------
# MODEL 3: ResNet
# -----------------------------------------------------------------------------
print("\n" + "-"*70)
print("MODEL 3: ResNet (Residual Network)")
print("-"*70)

class ResNetBlock(nn.Module):
    def __init__(self, dim, dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim, dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return x + self.net(x)

class ResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=4, dropout_rate=0.15):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([
            ResNetBlock(hidden_dim, dropout_rate) for _ in range(num_blocks)
        ])
        self.output_head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.output_head(x).squeeze(-1)

resnet_model = ResNet(X_train.shape[1], hidden_dim=256, num_blocks=4, dropout_rate=0.15).to(device)
optimizer = optim.AdamW(resnet_model.parameters(), lr=8e-4, weight_decay=5e-6)
criterion = nn.MSELoss()

start_time = time.time()
resnet_model = train_nn_model(resnet_model, train_loader, X_val_tensor, y_val_tensor, y_val_orig,
                             scaler_y, optimizer, criterion, n_epochs=500, patience=20)
resnet_time = time.time() - start_time

resnet_model.eval()
with torch.no_grad():
    y_pred_resnet_val_scaled = resnet_model(X_val_tensor).cpu().numpy()
    y_pred_resnet_test_scaled = resnet_model(X_test_tensor).cpu().numpy()

y_pred_resnet_val = scaler_y.inverse_transform(y_pred_resnet_val_scaled.reshape(-1, 1)).flatten()

resnet_metrics = calculate_metrics(y_val_orig, y_pred_resnet_val)

print(f"\n  ✓ Training time: {resnet_time:.1f}s")
print(f"    RMSE: {resnet_metrics['RMSE']:.4f}")
print(f"    MAPE: {resnet_metrics['MAPE']:.2f}%")
print(f"    R²:   {resnet_metrics['R2']:.4f}")

resnet_file = save_submission(y_pred_resnet_test_scaled, 'resnet', scaler_y)
results.append(['ResNet', resnet_metrics['RMSE'], resnet_metrics['MAPE'], resnet_metrics['R2'], resnet_time])

# Clear memory
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()



----------------------------------------------------------------------
MODEL 3: ResNet (Residual Network)
----------------------------------------------------------------------

  ✓ Training time: 56.2s
    RMSE: 0.1992
    MAPE: 21.61%
    R²:   0.8670
  ✓ Saved: submission_resnet_20260419_184337.csv


In [ ]:

# -----------------------------------------------------------------------------
# MODEL 4: FT-Transformer
# -----------------------------------------------------------------------------
print("\n" + "-"*70)
print("MODEL 4: FT-Transformer (Feature Tokenizer Transformer)")
print("-"*70)

class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_features, d_token))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight + self.bias

class TransformerBlock(nn.Module):
    def __init__(self, d_token, n_heads, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_token)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_token * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token * 2, d_token)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.dropout1(attn_out)
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class FTTransformer(nn.Module):
    def __init__(self, num_features, d_token=64, n_layers=3, n_heads=4, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, d_token)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_token, n_heads, dropout) for _ in range(n_layers)
        ])

        self.pred_head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.GELU(),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):
        tokens = self.tokenizer(x)
        batch_size = x.shape[0]
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        for block in self.blocks:
            tokens = block(tokens)

        return self.pred_head(tokens[:, 0, :]).squeeze(-1)

transformer_model = FTTransformer(
    num_features=X_train.shape[1],
    d_token=64,
    n_layers=3,
    n_heads=4,
    dropout=0.1
).to(device)

optimizer = optim.AdamW(transformer_model.parameters(), lr=5e-4, weight_decay=1e-5)
criterion = nn.MSELoss()

start_time = time.time()
transformer_model = train_nn_model(transformer_model, train_loader, X_val_tensor, y_val_tensor, y_val_orig,
                                  scaler_y, optimizer, criterion, n_epochs=500, patience=25)
transformer_time = time.time() - start_time

transformer_model.eval()
with torch.no_grad():
    y_pred_transformer_val_scaled = transformer_model(X_val_tensor).cpu().numpy()
    y_pred_transformer_test_scaled = transformer_model(X_test_tensor).cpu().numpy()

y_pred_transformer_val = scaler_y.inverse_transform(y_pred_transformer_val_scaled.reshape(-1, 1)).flatten()

transformer_metrics = calculate_metrics(y_val_orig, y_pred_transformer_val)

print(f"\n  ✓ Training time: {transformer_time:.1f}s")
print(f"    RMSE: {transformer_metrics['RMSE']:.4f}")
print(f"    MAPE: {transformer_metrics['MAPE']:.2f}%")
print(f"    R²:   {transformer_metrics['R2']:.4f}")

transformer_file = save_submission(y_pred_transformer_test_scaled, 'transformer', scaler_y)
results.append(['Transformer', transformer_metrics['RMSE'], transformer_metrics['MAPE'],
                transformer_metrics['R2'], transformer_time])





----------------------------------------------------------------------
MODEL 4: FT-Transformer (Feature Tokenizer Transformer)
----------------------------------------------------------------------


In [ ]:
# ============================================================================
# STEP 3: ENSEMBLE (WEIGHTED AVERAGE)
# ============================================================================
print("\n" + "="*70)
print("STEP 3: CREATING ENSEMBLE PREDICTION")
print("="*70)

# Calculate weights based on validation MAPE (lower MAPE = higher weight)
mapes = [lgb_metrics['MAPE'], mlp_metrics['MAPE'], resnet_metrics['MAPE'], transformer_metrics['MAPE']]
weights = [1/m for m in mapes]
weights = [w/sum(weights) for w in weights]

print(f"\n  Ensemble weights (based on validation MAPE):")
print(f"    LightGBM:  {weights[0]:.3f}")
print(f"    MLP:       {weights[1]:.3f}")
print(f"    ResNet:    {weights[2]:.3f}")
print(f"    Transformer: {weights[3]:.3f}")

# Weighted average
y_pred_ensemble_val = (weights[0] * y_pred_lgb_val +
                       weights[1] * y_pred_mlp_val +
                       weights[2] * y_pred_resnet_val +
                       weights[3] * y_pred_transformer_val)

y_pred_ensemble_test_scaled = (weights[0] * y_pred_lgb_test_scaled +
                               weights[1] * y_pred_mlp_test_scaled +
                               weights[2] * y_pred_resnet_test_scaled +
                               weights[3] * y_pred_transformer_test_scaled)

ensemble_metrics = calculate_metrics(y_val_orig, y_pred_ensemble_val)

print(f"\n  Ensemble validation metrics:")
print(f"    RMSE: {ensemble_metrics['RMSE']:.4f}")
print(f"    MAPE: {ensemble_metrics['MAPE']:.2f}%")
print(f"    R²:   {ensemble_metrics['R2']:.4f}")

ensemble_file = save_submission(y_pred_ensemble_test_scaled, 'ensemble_weighted', scaler_y)
results.append(['Ensemble (Weighted)', ensemble_metrics['RMSE'], ensemble_metrics['MAPE'],
                ensemble_metrics['R2'], 0])

# ============================================================================
# STEP 4: SUMMARY
# ============================================================================
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'MAPE (%)', 'R²', 'Time (s)'])
results_df = results_df.sort_values('MAPE (%)')

print("\n  Model Comparison (sorted by MAPE):")
print("  " + "-"*66)
print(f"  {'Model':<20} {'RMSE':>10} {'MAPE (%)':>12} {'R²':>10} {'Time (s)':>10}")
print("  " + "-"*66)
for _, row in results_df.iterrows():
    print(f"  {row['Model']:<20} {row['RMSE']:>10.4f} {row['MAPE (%)']:>12.2f} {row['R²']:>10.4f} {row['Time (s)']:>10.1f}")
print("  " + "-"*66)

# Best model
best_model = results_df.iloc[0]['Model']
best_mape = results_df.iloc[0]['MAPE (%)']
print(f"\n  ✓ BEST MODEL: {best_model} (MAPE: {best_mape:.2f}%)")

print("\n" + "="*70)
print("SUBMISSION FILES CREATED:")
print("="*70)
print(f"  1. {lgb_file}")
print(f"  2. {mlp_file}")
print(f"  3. {resnet_file}")
print(f"  4. {transformer_file}")
print(f"  5. {ensemble_file}")
print("="*70)

# Save results
results_df.to_csv('model_results_summary.csv', index=False)
print("\n  ✓ Results summary saved to: model_results_summary.csv")
print("="*70)

In [3]:
import pandas as pd
df=pd.read_csv('test(2).csv')
df

,Unnamed: 0,Tank Failure Pressure (bar),Liquid Ratio,Tank Width (m),Tank Length (m),Tank Height (m),BLEVE Height (m),Vapour Height (m),Vapour Temperature (K),Liquid Temperature (K),...,Obstacle Angle,Status,Liquid Critical Pressure (bar),Liquid Boiling Temperature (K),Liquid Critical Temperature (K),Sensor ID,Sensor Position Side,Sensor Position x,Sensor Position y,Sensor Position z
0,0,37.9,0.412227,0.8,6.6,0.4,0.8,0.2,317.6,337.5,...,0,Subcooled,42.5,-42,96.7,1,1,8.05,-4.3,-0.7
1,1,37.9,0.412227,0.8,6.6,0.4,0.8,0.2,317.6,337.5,...,0,Subcooled,42.5,-42,96.7,2,1,8.05,-4.3,5.1
2,2,37.9,0.412227,0.8,6.6,0.4,0.8,0.2,317.6,337.5,...,0,Subcooled,42.5,-42,96.7,3,1,8.05,-4.3,10.9
3,3,37.9,0.412227,0.8,6.6,0.4,0.8,0.2,317.6,337.5,...,0,Subcooled,42.5,-42,96.7,4,1,8.05,0.0,-0.7
4,4,37.9,0.412227,0.8,6.6,0.4,0.8,0.2,317.6,337.5,...,0,Subcooled,42.5,-42,96.7,5,1,8.05,0.0,5.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3198,3198,14.2,0.245940,1.6,8.6,1.8,1.0,1.4,457.3,422.7,...,1,Superheated,37.9,-1,152.0,13,2,19.75,0.4,-1.5
3199,3199,14.2,0.245940,1.6,8.6,1.8,1.0,1.4,457.3,422.7,...,1,Superheated,37.9,-1,152.0,14,2,19.75,0.4,0.8
3200,3200,14.2,0.245940,1.6,8.6,1.8,1.0,1.4,457.3,422.7,...,1,Superheated,37.9,-1,152.0,15,2,19.75,0.4,3.1
3201,3201,14.2,0.245940,1.6,8.6,1.8,1.0,1.4,457.3,422.7,...,1,Superheated,37.9,-1,152.0,16,2,19.75,5.7,-1.5
